# 01 — External Data Extraction (100% on Kaggle)
**Purpose:** Extract ALL external features directly on Kaggle.

**Part A:** API calls (SoilGrids, Elevation, Weather, OSM)

**Part B:** Heavy file download → extract per station → delete (`!wget → extract → !rm`)

**Output:** `train_enriched.parquet`, `val_enriched.parquet`

**Figures:**
- 🗺️ Data source coverage map (which stations got which data)
- 📊 Feature count bar chart (features per dataset source)
- 🔍 Extraction progress tracker

> ⚠️ **Enable Internet** in Kaggle notebook settings before running!

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import os
import time
import requests
from datetime import timedelta
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.figsize': (14, 6), 'font.size': 11, 'axes.titleweight': 'bold', 'figure.dpi': 120})

SEED = 42
INPUT_DIR = '/kaggle/input/ey-water-quality-data'
WORK_DIR = '/kaggle/working'
CHECKPOINT_DIR = f'{WORK_DIR}/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f'Checkpoint directory: {CHECKPOINT_DIR}')

In [ ]:
# Load base datasets from notebook 00 output
# If chaining notebooks: use the Kaggle Dataset output from notebook 00
train_base = pd.read_parquet(f'{WORK_DIR}/train_base.parquet')  # or INPUT_DIR
val_base = pd.read_parquet(f'{WORK_DIR}/val_base.parquet')

# Auto-detect columns
LAT_COL = [c for c in train_base.columns if 'lat' in c.lower()][0]
LON_COL = [c for c in train_base.columns if 'lon' in c.lower()][0]
DATE_COL = [c for c in train_base.columns if 'date' in c.lower()][0]
STATION_COL = [c for c in train_base.columns if 'station' in c.lower() or 'gems' in c.lower()][0]

print(f'Train: {train_base.shape}, Val: {val_base.shape}')
print(f'Cols: station={STATION_COL}, lat={LAT_COL}, lon={LON_COL}, date={DATE_COL}')

# Combine for extraction
all_data = pd.concat([train_base, val_base], ignore_index=True)
unique_stations = all_data[[STATION_COL, LAT_COL, LON_COL]].drop_duplicates()
print(f'Unique stations to extract: {len(unique_stations)}')

---
## Part A: API Extraction (Lightweight)

### A1. Elevation (OpenTopoData)

In [ ]:
def fetch_elevation(lat, lon, max_retries=3):
    url = 'https://api.opentopodata.org/v1/aster30m'
    for attempt in range(max_retries):
        try:
            r = requests.get(url, params={'locations': f'{lat},{lon}'}, timeout=30)
            r.raise_for_status()
            return float(r.json()['results'][0]['elevation'])
        except Exception:
            time.sleep(2 ** attempt)
    return np.nan

ckpt = f'{CHECKPOINT_DIR}/elevation.parquet'
if os.path.exists(ckpt):
    elev_df = pd.read_parquet(ckpt)
    print(f'✓ Loaded elevation checkpoint: {len(elev_df)} rows')
else:
    print(f'Extracting elevation for {len(unique_stations)} stations...')
    elev_df = unique_stations.copy()
    elev_df['elevation_m'] = elev_df.apply(lambda r: fetch_elevation(r[LAT_COL], r[LON_COL]), axis=1)
    elev_df.to_parquet(ckpt, index=False)
    print(f'✓ Saved: {len(elev_df)} rows, nulls={elev_df["elevation_m"].isnull().sum()}')

display(elev_df.describe())

### A2. SoilGrids (ISRIC)

In [ ]:
def fetch_soilgrids(lat, lon):
    properties = ['phh2o', 'clay', 'sand', 'silt', 'ocd', 'cec']
    results = {}
    for prop in properties:
        try:
            r = requests.get('https://rest.isric.org/soilgrids/v2.0/properties/query',
                           params={'lat': lat, 'lon': lon, 'property': prop, 'depth': '0-5cm', 'value': 'mean'},
                           timeout=30)
            r.raise_for_status()
            val = r.json()['properties']['layers'][0]['depths'][0]['values']['mean']
            results[f'soil_{prop}'] = float(val) if val else np.nan
        except Exception:
            results[f'soil_{prop}'] = np.nan
    return results

ckpt = f'{CHECKPOINT_DIR}/soilgrids.parquet'
if os.path.exists(ckpt):
    soil_df = pd.read_parquet(ckpt)
    print(f'✓ Loaded soilgrids checkpoint: {len(soil_df)} rows')
else:
    print(f'Extracting SoilGrids for {len(unique_stations)} stations...')
    soil_data = unique_stations.apply(lambda r: fetch_soilgrids(r[LAT_COL], r[LON_COL]), axis=1)
    soil_df = pd.concat([unique_stations.reset_index(drop=True), pd.DataFrame(soil_data.tolist())], axis=1)
    soil_df.to_parquet(ckpt, index=False)
    print(f'✓ Saved: {len(soil_df)} rows')

display(soil_df.describe())

### A3. Weather — Open-Meteo (7/14/30-day lag)

In [ ]:
def fetch_weather(lat, lon, date_str, lag_days=7):
    target_date = pd.to_datetime(date_str)
    start = (target_date - timedelta(days=lag_days)).strftime('%Y-%m-%d')
    end = target_date.strftime('%Y-%m-%d')
    try:
        r = requests.get('https://archive-api.open-meteo.com/v1/archive', params={
            'latitude': lat, 'longitude': lon, 'start_date': start, 'end_date': end,
            'daily': 'precipitation_sum,temperature_2m_max,temperature_2m_min,windspeed_10m_max',
            'timezone': 'Africa/Johannesburg'
        }, timeout=30)
        r.raise_for_status()
        data = r.json().get('daily', {})
        precip = [p for p in data.get('precipitation_sum', []) if p is not None]
        tmax = [t for t in data.get('temperature_2m_max', []) if t is not None]
        tmin = [t for t in data.get('temperature_2m_min', []) if t is not None]
        wind = [w for w in data.get('windspeed_10m_max', []) if w is not None]
        return {
            f'precip_sum_{lag_days}d': sum(precip) if precip else np.nan,
            f'precip_max_{lag_days}d': max(precip) if precip else np.nan,
            f'temp_max_{lag_days}d': max(tmax) if tmax else np.nan,
            f'temp_min_{lag_days}d': min(tmin) if tmin else np.nan,
            f'temp_range_{lag_days}d': (max(tmax) - min(tmin)) if tmax and tmin else np.nan,
            f'wind_avg_{lag_days}d': np.mean(wind) if wind else np.nan,
        }
    except Exception:
        return {k: np.nan for k in [f'precip_sum_{lag_days}d', f'precip_max_{lag_days}d',
                f'temp_max_{lag_days}d', f'temp_min_{lag_days}d', f'temp_range_{lag_days}d', f'wind_avg_{lag_days}d']}

weather_keys = all_data[[STATION_COL, LAT_COL, LON_COL, DATE_COL]].drop_duplicates()
print(f'Unique (station, date) combos for weather: {len(weather_keys)}')

In [ ]:
CHUNK_SIZE = 50
ckpt = f'{CHECKPOINT_DIR}/weather.parquet'

if os.path.exists(ckpt):
    weather_df = pd.read_parquet(ckpt)
    print(f'✓ Loaded weather checkpoint: {len(weather_df)} rows')
else:
    results = []
    total = len(weather_keys)
    
    for i in range(0, total, CHUNK_SIZE):
        chunk = weather_keys.iloc[i:i+CHUNK_SIZE].copy()
        for lag in [7, 14, 30]:
            chunk_weather = chunk.apply(lambda r: fetch_weather(r[LAT_COL], r[LON_COL], str(r[DATE_COL]), lag), axis=1)
            for col_name in chunk_weather.iloc[0].keys():
                chunk[col_name] = chunk_weather.apply(lambda x: x.get(col_name, np.nan))
        results.append(chunk)
        
        pct = min(100, (i+CHUNK_SIZE)/total*100)
        print(f'  Weather progress: {pct:.0f}% ({min(i+CHUNK_SIZE, total)}/{total})')
        
        if len(results) % 10 == 0:  # checkpoint every 500 rows
            pd.concat(results, ignore_index=True).to_parquet(f'{CHECKPOINT_DIR}/weather_partial.parquet', index=False)
    
    weather_df = pd.concat(results, ignore_index=True)
    weather_df.to_parquet(ckpt, index=False)
    print(f'✓ Saved weather: {len(weather_df)} rows')

display(weather_df.describe())

### A4. OSM Pollution Proximity

In [ ]:
def fetch_osm_counts(lat, lon, radius_m=5000):
    query = f"""[out:json][timeout:30];
    (node(around:{radius_m},{lat},{lon})["man_made"="mine"];
     way(around:{radius_m},{lat},{lon})["man_made"="wastewater_plant"];
     way(around:{radius_m},{lat},{lon})["landuse"="farmland"];
     way(around:{radius_m},{lat},{lon})["highway"];);
    out count;"""
    try:
        r = requests.get('http://overpass-api.de/api/interpreter', params={'data': query}, timeout=60)
        r.raise_for_status()
        total = r.json().get('elements', [{}])[0].get('tags', {}).get('total', 0)
        return {f'osm_total_{radius_m}m': int(total)}
    except Exception:
        return {f'osm_total_{radius_m}m': 0}

ckpt = f'{CHECKPOINT_DIR}/osm.parquet'
if os.path.exists(ckpt):
    osm_df = pd.read_parquet(ckpt)
    print(f'✓ Loaded OSM checkpoint: {len(osm_df)} rows')
else:
    print(f'Extracting OSM for {len(unique_stations)} stations (3 radii)...')
    osm_df = unique_stations.copy().reset_index(drop=True)
    for radius in [1000, 5000, 10000]:
        osm_data = osm_df.apply(lambda r: fetch_osm_counts(r[LAT_COL], r[LON_COL], radius), axis=1)
        for col in osm_data.iloc[0].keys():
            osm_df[col] = osm_data.apply(lambda x: x.get(col, 0))
        print(f'  Radius {radius}m done')
    osm_df.to_parquet(ckpt, index=False)
    print(f'✓ Saved: {len(osm_df)} rows')

display(osm_df.describe())

---
## Part B: Heavy File Download → Extract → Delete

### B1. HydroATLAS / B2. RiverATLAS / B3. SANLC / B4. WorldPop

> These use the `!wget → extract → !rm` pattern. 
> Fill in the download URLs when ready.

In [ ]:
# Placeholder for heavy datasets — fill with actual download URLs
# Pattern: !wget URL → geopandas/rasterio extract per station → save parquet → !rm big file

heavy_datasets = ['hydroatlas', 'riveratlas', 'sanlc', 'worldpop']
heavy_dfs = {}

for ds in heavy_datasets:
    ckpt = f'{CHECKPOINT_DIR}/{ds}.parquet'
    if os.path.exists(ckpt):
        heavy_dfs[ds] = pd.read_parquet(ckpt)
        print(f'✓ Loaded {ds} checkpoint: {len(heavy_dfs[ds])} rows')
    else:
        print(f'⚠️ {ds} not extracted yet — add download URL and extraction code')
        heavy_dfs[ds] = unique_stations.copy()
        heavy_dfs[ds][f'{ds}_placeholder'] = np.nan
        heavy_dfs[ds].to_parquet(ckpt, index=False)

---
## Merge All External Features

In [ ]:
station_keys = [STATION_COL, LAT_COL, LON_COL]
external_static = unique_stations.copy()

all_source_dfs = [
    (elev_df, 'elevation'), (soil_df, 'soilgrids'), (osm_df, 'osm'),
] + [(heavy_dfs[ds], ds) for ds in heavy_datasets]

feature_counts = {}  # for plotting

for feat_df, name in all_source_dfs:
    new_cols = [c for c in feat_df.columns if c not in station_keys]
    if new_cols:
        merge_df = feat_df[station_keys + new_cols].copy()
        external_static = external_static.merge(merge_df, on=station_keys, how='left')
        feature_counts[name] = len(new_cols)
        print(f'  + {name}: {len(new_cols)} features → total: {external_static.shape[1]}')

print(f'\nStatic features: {external_static.shape}')

In [ ]:
# Merge static + temporal (weather)
train_enriched = train_base.merge(external_static, on=station_keys, how='left')
val_enriched = val_base.merge(external_static, on=station_keys, how='left')

weather_merge_keys = [STATION_COL, LAT_COL, LON_COL, DATE_COL]
weather_new_cols = [c for c in weather_df.columns if c not in weather_merge_keys]
feature_counts['weather'] = len(weather_new_cols)

train_enriched = train_enriched.merge(weather_df[weather_merge_keys + weather_new_cols], on=weather_merge_keys, how='left')
val_enriched = val_enriched.merge(weather_df[weather_merge_keys + weather_new_cols], on=weather_merge_keys, how='left')

print(f'Train enriched: {train_enriched.shape}')
print(f'Val enriched: {val_enriched.shape}')
print(f'New features added: {train_enriched.shape[1] - train_base.shape[1]}')

## 📊 FIGURE 1: Features Added per Data Source

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

sources = list(feature_counts.keys())
counts = list(feature_counts.values())
colors_bar = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336', '#00BCD4', '#795548', '#607D8B']

bars = ax.bar(sources, counts, color=colors_bar[:len(sources)], edgecolor='white', linewidth=1.5)
ax.bar_label(bars, fontsize=11, fontweight='bold')

ax.set_ylabel('Number of Features', fontsize=12)
ax.set_title('📊 Features Added by Each External Data Source\n'
             f'Total external features: {sum(counts)}', fontsize=13)
ax.set_xlabel('Data Source')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(f'{WORK_DIR}/fig_01_features_per_source.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: fig_01_features_per_source.png')

## 📊 FIGURE 2: Data Coverage Map (Which Stations Got Data)

In [ ]:
# Show which data sources have valid (non-null) data per station
coverage_cols = {
    'Elevation': 'elevation_m',
    'SoilGrids': 'soil_phh2o',
    'OSM': 'osm_total_5000m',
}

n_sources = len(coverage_cols) + 1  # +1 for weather
fig, axes = plt.subplots(1, n_sources, figsize=(5*n_sources, 6))

for i, (source_name, col_name) in enumerate(coverage_cols.items()):
    ax = axes[i]
    if col_name in external_static.columns:
        has_data = external_static[col_name].notna()
        colors_map = ['#4CAF50' if v else '#F44336' for v in has_data]
        ax.scatter(external_static[LON_COL], external_static[LAT_COL],
                  c=colors_map, s=40, alpha=0.8, edgecolors='gray', linewidths=0.3)
        n_ok = has_data.sum()
        ax.set_title(f'{source_name}\n{n_ok}/{len(has_data)} stations', fontsize=11)
    else:
        ax.set_title(f'{source_name}\nNot extracted', fontsize=11)
    ax.set_xlim(16, 33); ax.set_ylim(-35, -22)
    ax.set_xlabel('Lon'); ax.set_ylabel('Lat')

# Weather coverage (per sample, not per station)
ax = axes[-1]
weather_col = [c for c in weather_df.columns if 'precip' in c.lower()]
if weather_col:
    has_weather = weather_df[weather_col[0]].notna()
    n_ok = has_weather.sum()
    ax.set_title(f'Weather\n{n_ok}/{len(has_weather)} samples', fontsize=11)
    ax.text(0.5, 0.5, f'{n_ok}/{len(has_weather)}\nsamples\ncovered', 
            transform=ax.transAxes, ha='center', va='center', fontsize=14)
ax.set_xlim(16, 33); ax.set_ylim(-35, -22)

# Legend
legend_elements = [mpatches.Patch(facecolor='#4CAF50', label='Has data'),
                   mpatches.Patch(facecolor='#F44336', label='Missing')]
fig.legend(handles=legend_elements, loc='lower center', ncol=2, fontsize=10)

fig.suptitle('🗺️ Data Coverage per Source (green=OK, red=missing)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{WORK_DIR}/fig_01_data_coverage_map.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: fig_01_data_coverage_map.png')

## 📊 FIGURE 3: Extraction Summary Table

In [ ]:
# Summary table of all extraction results
summary_data = []
for source_name, col_name in coverage_cols.items():
    if col_name in external_static.columns:
        n_valid = external_static[col_name].notna().sum()
        n_total = len(external_static)
        summary_data.append({'Source': source_name, 'Features': feature_counts.get(source_name.lower(), 0),
                            'Coverage': f'{n_valid}/{n_total}', 'Pct': f'{n_valid/n_total*100:.0f}%',
                            'Status': '✅' if n_valid == n_total else '⚠️'})

summary_data.append({'Source': 'Weather', 'Features': feature_counts.get('weather', 0),
                     'Coverage': f'{len(weather_df)}/{len(weather_keys)}', 
                     'Pct': f'{len(weather_df)/max(1,len(weather_keys))*100:.0f}%',
                     'Status': '✅'})

for ds in heavy_datasets:
    summary_data.append({'Source': ds.title(), 'Features': feature_counts.get(ds, 0),
                        'Coverage': 'Placeholder', 'Pct': '0%', 'Status': '⚠️ TODO'})

summary_table = pd.DataFrame(summary_data)
display(summary_table)

## Save Enriched Datasets

In [ ]:
train_enriched.to_parquet(f'{WORK_DIR}/train_enriched.parquet', index=False)
val_enriched.to_parquet(f'{WORK_DIR}/val_enriched.parquet', index=False)

print(f'✅ Saved train_enriched.parquet: {train_enriched.shape}')
print(f'✅ Saved val_enriched.parquet: {val_enriched.shape}')
print(f'\nNew features: {train_enriched.shape[1] - train_base.shape[1]}')
print(f'\n📊 Figures saved:')
print(f'  1. fig_01_features_per_source.png — Features per data source')
print(f'  2. fig_01_data_coverage_map.png — Coverage map per source')